In [1]:
! pip install requests beautifulsoup4 selenium

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import uuid
import time
import random
import re

# Настройка Chrome с реалистичными параметрами
options = Options()
# options.add_argument("--headless")  # Раскомментируйте для запуска в фоновом режиме после отладки
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--window-size=1920,1080")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/112.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

product_urls = {
    "зубная паста": "https://kaspi.kz/shop/p/splat-zubnaja-pasta-ul-trakompleks-100-ml-100194794/?c=750000000",
    "гель для бровей": "https://kaspi.kz/shop/p/pusy-gel-rozovyi-super-fix-clear-5-ml-119031367/?c=750000000",
    "кисть для тона": "https://kaspi.kz/shop/p/kist-dlja-tonal-nogo-krema-serebrjanyi-1-117403333/?c=750000000",
    "масло для волос": "https://kaspi.kz/shop/p/tashe-liquid-silk-maslo-100-ml-117503489/?c=750000000",
    "расческа": "https://kaspi.kz/shop/p/hollow-comb-massazhnaja-rascheska-21-sm-102186204/?c=750000000"
}

def human_like_scroll(driver):
    scroll_height = driver.execute_script("return document.body.scrollHeight")
    current_position = driver.execute_script("return window.pageYOffset")
    
    while current_position < scroll_height - 500:
        step = random.randint(300, 700)
        driver.execute_script(f"window.scrollBy(0, {step});")
        time.sleep(random.uniform(0.2, 0.5))
        current_position = driver.execute_script("return window.pageYOffset")
        
        if random.random() > 0.7:
            scroll_height = driver.execute_script("return document.body.scrollHeight")

def click_show_more(driver):
    human_like_scroll(driver)
    time.sleep(1)
    
    try:
        show_more_xpaths = [
            "//button[contains(., 'Показать ещё')]",
            "//button[contains(., 'Показать еще')]",
            "//div[contains(@class, 'show-more')]//button",
            "//div[contains(@class, 'load-more')]//button",
            "//a[contains(., 'Показать ещё')]",
            "//span[contains(., 'Показать ещё')]/parent::*"
        ]
        
        for xpath in show_more_xpaths:
            try:
                button = WebDriverWait(driver, 3).until(
                    EC.element_to_be_clickable((By.XPATH, xpath))
                )
                print(f"Найдена кнопка с XPath: {xpath}")
                driver.execute_script("arguments[0].click();", button)
                time.sleep(random.uniform(3, 5))
                return True
            except:
                continue
                
    except Exception as e:
        print(f"Ошибка при попытке клика на кнопку: {e}")
    
    # Пробуем прокрутку для бесконечной загрузки
    try:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        
        for _ in range(3):
            driver.execute_script("window.scrollBy(0, 100);")
            time.sleep(1)
    except Exception as e:
        print(f"Ошибка при эмуляции прокрутки: {e}")
    
    return False

all_reviews = []

for product_name, url in product_urls.items():
    print(f"\nОбрабатываем {product_name}: {url}")
    driver.get(url)
    time.sleep(random.uniform(5, 8))
    
    # Проверяем наличие отзывов на странице или ищем ссылку на отзывы
    try:
        review_links = driver.find_elements(By.XPATH, "//a[contains(@href, 'review') or contains(@href, 'отзыв') or contains(text(), 'Отзыв')]")
        if review_links:
            for link in review_links:
                if "отзыв" in link.text.lower() or "review" in link.text.lower():
                    driver.execute_script("arguments[0].click();", link)
                    print("Перешли по ссылке на отзывы")
                    time.sleep(3)
                    break
        else:
            print("Не удалось найти ссылку на отзывы, пробуем продолжить с текущей страницы")
    except Exception as e:
        print(f"Ошибка при поиске ссылки на отзывы: {e}")
        print("Продолжаем с текущей страницы")
    
    human_like_scroll(driver)
    time.sleep(2)
    
    collected = 0
    processed_reviews = set()
    retry_count = 0
    
    while collected < 100 and retry_count < 5:
        soup = BeautifulSoup(driver.page_source, "html.parser")
        reviews = soup.find_all("div", class_="reviews__review")
        
        if not reviews:
            print(f"Нет отзывов для {product_name}")
            break
        
        print(f"Найдено {len(reviews)} отзывов на текущей странице для {product_name}")
        
        new_reviews_found = False
        for review in reviews:
            review_content = review.get_text(strip=True)
            review_hash = hash(review_content)
            
            if review_hash not in processed_reviews and review_content.strip():
                processed_reviews.add(review_hash)
                new_reviews_found = True
                
                review_id = str(uuid.uuid4())
                
                date_tag = review.find("div", class_="reviews__date")
                date = date_tag.text.strip() if date_tag and date_tag.text else None
                
                comment_tag = review.find("div", class_="reviews__review-text")
                comment = None
                if comment_tag:
                    comment_text = comment_tag.get_text(strip=True)
                    comment = re.sub(r'^Комментарий:\s*', '', comment_text).strip()
                
                rating_tag = review.find("div", class_=lambda c: c and isinstance(c, str) and "rating" in c)
                rating = None
                if rating_tag:
                    classes = rating_tag.get("class", [])
                    for cls in classes:
                        if isinstance(cls, str) and cls.startswith("_") and cls[1:].isdigit():
                            try:
                                rating = int(cls[1:]) / 10
                            except ValueError:
                                rating = None
                
                all_reviews.append({
                    "ID": review_id,
                    "Атауы": product_name,
                    "Күні": date,
                    "Баға": rating,
                    "Пікір": comment
                })
                
                collected += 1
                if collected >= 100:
                    break
        
        if collected >= 100:
            print(f"Собрали 100 отзывов для {product_name}, переходим к следующему продукту")
            break
        
        if not new_reviews_found:
            retry_count += 1
            print(f"Не найдено новых отзывов. Попытка {retry_count} из 5")
            
            if retry_count >= 5:
                print("Превышено количество попыток")
                break
        else:
            retry_count = 0
        
        if not click_show_more(driver):
            retry_count += 1
            print(f"Не удалось загрузить больше отзывов. Попытка {retry_count} из 5")
            time.sleep(2)
            
            if retry_count >= 5:
                print("Превышено количество попыток загрузки дополнительных отзывов")
                break
    
    print(f"Собрано {collected} отзывов для {product_name}")

driver.quit()

df = pd.DataFrame(all_reviews)
df.to_csv("kaspi_reviews.csv", index=False, encoding='utf-8-sig')
print(f"{len(all_reviews)} пікір сәтті сақталды!")


Обрабатываем зубная паста: https://kaspi.kz/shop/p/splat-zubnaja-pasta-ul-trakompleks-100-ml-100194794/?c=750000000
Перешли по ссылке на отзывы
Найдено 40 отзывов на текущей странице для зубная паста
Найдена кнопка с XPath: //a[contains(., 'Показать ещё')]
Найдено 140 отзывов на текущей странице для зубная паста
Собрали 100 отзывов для зубная паста, переходим к следующему продукту
Собрано 100 отзывов для зубная паста

Обрабатываем гель для бровей: https://kaspi.kz/shop/p/pusy-gel-rozovyi-super-fix-clear-5-ml-119031367/?c=750000000
Найдено 45 отзывов на текущей странице для гель для бровей
Найдена кнопка с XPath: //a[contains(., 'Показать ещё')]
Найдено 189 отзывов на текущей странице для гель для бровей
Собрали 100 отзывов для гель для бровей, переходим к следующему продукту
Собрано 100 отзывов для гель для бровей

Обрабатываем кисть для тона: https://kaspi.kz/shop/p/kist-dlja-tonal-nogo-krema-serebrjanyi-1-117403333/?c=750000000
Найдено 27 отзывов на текущей странице для кисть для то

In [3]:
df

,ID,Атауы,Күні,Баға,Пікір
0,c3036984-6b38-4c13-a268-43f3af18c65c,зубная паста,03.05.2025,5.0,"Хорошая паста, я давно ей пользуюсь."
1,90ba58c4-25ae-4709-9221-ae9567c08d7d,зубная паста,02.05.2025,5.0,Пользуюсь пастой SPLAT Ультракомплекс уже не п...
2,84bacc27-f592-49fa-a61e-ee5e72a5ba7b,зубная паста,02.05.2025,5.0,"Всегда берём эту пасту, очень нравится! Соотно..."
3,72260e6a-c4e1-47e5-bd33-598f8d664de9,зубная паста,02.05.2025,5.0,Зубная паста такая вкусная.
4,7149a797-57d1-407b-baec-e2a90b641f1e,зубная паста,02.05.2025,5.0,Работаю в Автомойке. Уже этими тряпками 50 маш...
...,...,...,...,...,...
495,f2880dce-9c7f-44bd-8224-28a5fd951a02,расческа,10.11.2024,5.0,Классная расчёска! Именно для нарощенных волос...
496,1c2cdc0a-21df-4813-b34e-9871c1b1a928,расческа,09.11.2024,5.0,Расчёска просто находка. Спасибо магазину Hub....
497,c82f73b0-07bb-4a75-adcc-d98d2a3ae80c,расческа,07.11.2024,5.0,"Хорошая расческа, дочке понравилась. Как на ка..."
498,1b7bd45d-fd40-46ee-8784-601ccf9abcee,расческа,23.10.2024,5.0,"Обычная расческа нормальная, пришел другой цве..."
